# 02 — Market Intelligence: 10 Business SQL Queries

Using `pandasql` to run SQL on in-memory Google Trends data.

**Dataset:** Real Google Trends (pytrends API) — 14 keywords, 5 years, worldwide + US regional.


In [ ]:
import pandas as pd
import pandasql as ps
import numpy as np
from datetime import datetime

df_ww = pd.read_csv('data/interest_over_time_worldwide.csv', index_col=0, parse_dates=True).reset_index()
df_ww.columns = ['date'] + list(df_ww.columns[1:])
df_us = pd.read_csv('data/interest_over_time_us.csv', index_col=0, parse_dates=True).reset_index()
df_us.columns = ['date'] + list(df_us.columns[1:])
df_region = pd.read_csv('data/interest_by_region_us.csv')

# Melt to long format for SQL
ww_long = df_ww.melt(id_vars=['date'], var_name='keyword', value_name='interest')
us_long = df_us.melt(id_vars=['date'], var_name='keyword', value_name='interest')

print(f'ww_long: {ww_long.shape}, us_long: {us_long.shape}, region: {df_region.shape}')

## Q1 — Topic Interest Ranking by Volume & Growth Rate

In [ ]:
q1 = '''
SELECT 
    keyword,
    ROUND(AVG(interest), 2) as avg_interest,
    ROUND(MAX(interest), 2) as peak_interest,
    ROUND(AVG(CASE WHEN date >= date('now', '-1 year') THEN interest END), 2) as recent_avg,
    ROUND(AVG(CASE WHEN date >= date('now', '-2 year') AND date < date('now', '-1 year') THEN interest END), 2) as prior_avg
FROM ww_long
GROUP BY keyword
ORDER BY avg_interest DESC
'''
print(ps.sqldf(q1, locals()).to_string(index=False))

## Q2 — Regional Interest Heatmap (top state per topic)

In [ ]:
q2 = '''
SELECT keyword, geoName, interest
FROM (
    SELECT keyword, geoName, interest,
           ROW_NUMBER() OVER (PARTITION BY keyword ORDER BY interest DESC) as rn
    FROM df_region
)
WHERE rn = 1
ORDER BY interest DESC
'''
print(ps.sqldf(q2, locals()).to_string(index=False))

## Q3 — Emerging Topic Detection (growth >100% YoY)

In [ ]:
# Compute YoY growth in a temp table
ww_long['year'] = ww_long['date'].dt.year
yearly = ww_long.groupby(['keyword', 'year'])['interest'].mean().reset_index()
yearly['prior'] = yearly.groupby('keyword')['interest'].shift(1)
yearly['growth_pct'] = ((yearly['interest'] - yearly['prior']) / yearly['prior'] * 100).round(1)

q3 = '''
SELECT keyword, year, ROUND(interest, 2) as avg_interest, growth_pct
FROM yearly
WHERE growth_pct > 100 AND year >= 2022
ORDER BY growth_pct DESC
LIMIT 15
'''
print(ps.sqldf(q3, locals()).to_string(index=False))

## Q4 — Trend Correlation Matrix (SQL pivot)

In [ ]:
# Self-join to compute correlations
corr_pairs = []
keywords = ww_long['keyword'].unique()
for i, a in enumerate(keywords):
    for b in keywords[i+1:]:
        merged = ww_long[ww_long['keyword'] == a][['date', 'interest']].merge(
            ww_long[ww_long['keyword'] == b][['date', 'interest']], on='date', suffixes=('_a', '_b')
        )
        if len(merged) > 10:
            r = np.corrcoef(merged['interest_a'], merged['interest_b'])[0, 1]
            corr_pairs.append({'topic_a': a, 'topic_b': b, 'correlation': round(r, 3)})
corr_df = pd.DataFrame(corr_pairs)
print(corr_df.sort_values('correlation', ascending=False).head(15).to_string(index=False))

## Q5 — Seasonal Pattern Detection (monthly aggregation)

In [ ]:
ww_long['month'] = ww_long['date'].dt.month
seasonal = ww_long.groupby(['keyword', 'month'])['interest'].mean().reset_index()

q5 = '''
SELECT keyword, month, ROUND(interest, 2) as avg_interest
FROM seasonal
WHERE keyword IN ('fitness', 'mental health', 'crypto', 'AI')
ORDER BY keyword, month
'''
print(ps.sqldf(q5, locals()).to_string(index=False))

## Q6 — Event-Driven Spike Analysis (weeks with >3σ moves)

In [ ]:
# Z-score based spike detection
spike_data = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date')
    sub['rolling_mean'] = sub['interest'].rolling(12).mean()
    sub['rolling_std'] = sub['interest'].rolling(12).std()
    sub['zscore'] = (sub['interest'] - sub['rolling_mean']) / sub['rolling_std']
    spikes = sub[sub['zscore'] > 3]
    for _, row in spikes.iterrows():
        spike_data.append({'keyword': kw, 'date': row['date'].strftime('%Y-%m-%d'), 'interest': row['interest'], 'zscore': round(row['zscore'], 2)})
spike_df = pd.DataFrame(spike_data)
print(spike_df.head(15).to_string(index=False) if len(spike_df) > 0 else 'No extreme spikes (>3σ) detected in this window.')

## Q7 — Category Lifecycle: Early Growth → Peak → Decline

In [ ]:
lifecycle = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date')
    peak_idx = sub['interest'].idxmax()
    peak_date = sub.loc[peak_idx, 'date']
    peak_val = sub.loc[peak_idx, 'interest']
    early = sub[sub['date'] < peak_date]['interest'].mean() if (sub['date'] < peak_date).any() else 0
    late = sub[sub['date'] > peak_date]['interest'].mean() if (sub['date'] > peak_date).any() else 0
    lifecycle.append({'keyword': kw, 'peak_date': peak_date.strftime('%Y-%m-%d'), 'peak_interest': peak_val,
                      'early_avg': round(early, 1), 'late_avg': round(late, 1),
                      'trend': 'Rising' if late > early * 1.1 else ('Declining' if late < early * 0.9 else 'Stable')})
lifecycle_df = pd.DataFrame(lifecycle).sort_values('peak_date')
print(lifecycle_df.to_string(index=False))

## Q8 — Cross-Category Opportunity (low competition, high growth)

In [ ]:
# Compute mean (proxy for competition) and recent YoY growth
opp = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date')
    mean_i = sub['interest'].mean()
    recent = sub['interest'].iloc[-52:].mean()
    prior = sub['interest'].iloc[-104:-52].mean() if len(sub) >= 104 else sub['interest'].iloc[:52].mean()
    growth = ((recent - prior) / prior * 100) if prior > 0 else 0
    opp.append({'keyword': kw, 'mean_interest': round(mean_i, 1), 'recent_growth_pct': round(growth, 1)})
opp_df = pd.DataFrame(opp)
opp_df['opportunity_score'] = (opp_df['recent_growth_pct'] / (opp_df['mean_interest'] + 1)).round(2)
print(opp_df.sort_values('opportunity_score', ascending=False).to_string(index=False))

## Q9 — Interest Forecasting (Simple Linear Projection)

In [ ]:
from sklearn.linear_model import LinearRegression

forecast_results = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date').dropna()
    X = np.arange(len(sub)).reshape(-1, 1)
    y = sub['interest'].values
    model = LinearRegression().fit(X, y)
    future_X = np.array([[len(sub) + i] for i in range(1, 53)])  # 1 year ahead
    preds = model.predict(future_X)
    trend_slope = model.coef_[0]
    forecast_results.append({'keyword': kw, 'trend_slope': round(trend_slope, 3),
                             'forecast_next_yr_avg': round(preds.mean(), 1),
                             'trend_direction': 'Up' if trend_slope > 0.05 else ('Down' if trend_slope < -0.05 else 'Flat')})
forecast_df = pd.DataFrame(forecast_results).sort_values('trend_slope', ascending=False)
print(forecast_df.to_string(index=False))

## Q10 — Geographic Arbitrage (topics hot in one state, cold in another)

In [ ]:
# For each keyword, find max state and min state (with non-zero interest)
arbitrage = []
for kw in df_region['keyword'].unique():
    sub = df_region[(df_region['keyword'] == kw) & (df_region['interest'] > 0)]
    if len(sub) > 1:
        max_row = sub.loc[sub['interest'].idxmax()]
        min_row = sub.loc[sub['interest'].idxmin()]
        ratio = max_row['interest'] / min_row['interest'] if min_row['interest'] > 0 else float('inf')
        arbitrage.append({'keyword': kw, 'hottest_state': max_row['geoName'], 'hottest_interest': max_row['interest'],
                          'coldest_state': min_row['geoName'], 'coldest_interest': min_row['interest'],
                          'arbitrage_ratio': round(ratio, 1)})
arb_df = pd.DataFrame(arbitrage).sort_values('arbitrage_ratio', ascending=False)
print(arb_df.to_string(index=False))